# Lesson 01: Learning and Data Scarcity - Full Solution Notebook

Course: Machine Learning for Chemical Engineering

Focus: bias, variance, scarce experiments, and physically impossible extrapolation.

## Learning goals
1. Generate a small noisy reactor-yield dataset.
2. Fit polynomial models of different degrees.
3. Compare training error and test error.
4. Identify which model underfits, which overfits, and which is physically plausible.

## Colab use
Run cells from top to bottom. The notebook is self-contained and uses synthetic data so it can run from GitHub/Colab without external files.

## Full worked solution
This version contains one possible complete implementation. Use it after attempting the exercise notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(7)

def true_yield(T):
    """Synthetic yield curve with an optimum and degradation at high temperature."""
    return 1.8 * np.exp(-0.5 * ((T - 345) / 13)**2) + 0.25

T_train = np.linspace(310, 365, 10)
y_train = true_yield(T_train) + rng.normal(0, 0.08, size=T_train.shape)
T_test = np.linspace(305, 375, 300)
y_test = true_yield(T_test)

degrees = [1, 3, 9]
models = {}
scores = []

for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(T_train.reshape(-1, 1), y_train)
    train_pred = model.predict(T_train.reshape(-1, 1))
    test_pred = model.predict(T_test.reshape(-1, 1))
    models[degree] = test_pred
    scores.append((degree, mean_squared_error(y_train, train_pred), mean_squared_error(y_test, test_pred)))

print("degree | train MSE | test MSE")
for degree, train_mse, test_mse in scores:
    print(f"{degree:6d} | {train_mse:9.4f} | {test_mse:8.4f}")

plt.figure(figsize=(8, 4.5))
plt.plot(T_test, y_test, "k--", label="true physical response")
plt.scatter(T_train, y_train, c="tab:blue", label="scarce experiments")
for degree, pred in models.items():
    plt.plot(T_test, pred, label=f"degree {degree}")
plt.xlabel("Temperature [K]")
plt.ylabel("Yield [a.u.]")
plt.ylim(-0.5, 2.4)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print("\nInterpretation:")
print("Degree 1 underfits the curved physics. Degree 9 nearly memorizes the 10 points and can oscillate.")
print("The useful model is not the one with the smallest training error; it is the one that generalizes and stays physically plausible.")

## Reflection

Use the numerical result and the plots to answer:

1. What physical assumption did the model encode?
2. Where could the model fail outside the sampled domain?
3. Which validation metric would matter most in a real engineering deployment?